In [ ]:
pip uninstall -y protobuf tensorflow google-ai-generativelanguage

Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Found existing installation: google-ai-generativelanguage 0.6.15
Uninstalling google-ai-generativelanguage-0.6.15:
  Successfully uninstalled google-ai-generativelanguage-0.6.15


In [4]:
pip install -q vllm pyngrok huggingface_hub openai

In [2]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get('HF_TOKEN')
if token:
    login(token)

In [3]:
import os

os.makedirs("./models", exist_ok=True)

In [5]:
!hf download Qwen/Qwen2.5-1.5B-Instruct-AWQ --local-dir ./models

Fetching 10 files:   0% 0/10 [00:00<?, ?it/s]Downloading 'model.safetensors' to 'models/.cache/huggingface/download/xGOKKLRSlIhH692hSVvI1-gpoa8=.5b29ed6f80e422c3ae7d40dc76fd8983501bbd7489cf84a72a5c5fe20bcf2813.incomplete'

merges.txt: 0.00B [00:00, ?B/s]Downloading '.gitattributes' to 'models/.cache/huggingface/download/wPaCkH-WbT7GsmxMKKrNZTV4nSM=.39e7ae7fd0fdd2d8e5bc370225bb1f3eb8648ac8.incomplete'


config.json: 100% 838/838 [00:00<00:00, 6.63MB/s]
Download complete. Moving file to models/config.json


.gitattributes: 1.52kB [00:00, 557kB/s]
Download complete. Moving file to models/.gitattributes
Fetching 10 files:  10% 1/10 [00:00<00:01,  4.73it/s]

LICENSE: 11.3kB [00:00, 30.9MB/s]
Download complete. Moving file to models/LICENSE



tokenizer.json: 0.00B [00:00, ?B/s]



README.md: 5.28kB [00:00, 19.9MB/s]
Download complete. Moving file to models/README.md


generation_config.json: 100% 242/242 [00:00<00:00, 1.77MB/s]
Download complete. Moving file to models/generation_config.json

In [ ]:
# Redirection Breakdown: > vllm_server.log 2>&1
# ---------------------------------------------------------
# 1. > vllm_server.log : Redirects STDOUT (1) to the log file (overwrites).
# 2. 2>&1              : Redirects STDERR (2) to the same place as STDOUT (1).
#
# WHY: Captures both "Model loaded" and "Out of Memory" errors in one file.
# NOTE: Order matters! Must define file destination BEFORE 2>&1.
# TIP: Use '>>' instead of '>' to append rather than overwrite.

In [ ]:
import subprocess
import os

# Open a log file
log_file = open("vllm_server.log", "w")

# Start vLLM in a completely detached process
process = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "./models",
    "--quantization", "awq",
    "--dtype", "half",
    "--max-model-len", "2048",
    "--port", "8000",
    "--host", "0.0.0.0",
    "--enforce-eager",
    "--trust-remote-code",
    "--gpu-memory-utilization", "0.7"
], stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)

print(f"vLLM server started in background with PID: {process.pid}")
print("Run the next cell to check the logs.")


In [ ]:
!tail -f vllm_server.log

(APIServer pid=4277) INFO 03-17 06:53:57 [launcher.py:47] Route: /scale_elastic_ep, Methods: POST
(APIServer pid=4277) INFO 03-17 06:53:57 [launcher.py:47] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=4277) INFO:     Started server process [4277]
(APIServer pid=4277) INFO:     Waiting for application startup.
(APIServer pid=4277) INFO:     Application startup complete.
(APIServer pid=4277) INFO 03-17 06:54:11 [launcher.py:122] Shutting down FastAPI HTTP server.
[rank0]:[W317 06:54:11.637644003 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
(APIServer pid=4277) INFO:     Shutting down
(APIServer pid=4277) INFO:     Waiting for application shutdown.
(APIServer pid=4277) INFO:     Application shutdown complete.


In [8]:
from pyngrok import ngrok, conf

NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

ngrok.connect(8000).public_url

In [9]:
ngrok.connect(8000).public_url

'https://32d4-34-11-202-109.ngrok-free.app'

In [ ]:
ngrok.connect(8000)

<NgrokTunnel: "https://20be-136-117-97-86.ngrok-free.app" -> "http://localhost:8000">